# LegalQA v9.2 — grouped-dev candidate oracle và selector calibration
Attach source v9.2, official data, bốn approved base models, completed global dense index và QLoRA `split` artifact. Notebook dùng `selector_train.json` cho kNN nên 400 reference dev không lọt vào train-memory. Agentic được gate và chấm trên candidate-specific evidence.

In [ ]:
TRIALS = 3000
WORKING = '/kaggle/working/legalqa_selector_calibration'
AUTO_RESUME = True  # Attach đúng một selector_recovery_latest.zip để tiếp tục session trước


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
def require_kaggle_cuda():
    nvidia_smi = shutil.which('nvidia-smi')
    if nvidia_smi is None:
        raise RuntimeError('Session Kaggle chưa được gắn GPU. Vào Settings/Session options → Accelerator → GPU (T4 x2 hoặc T4 x1), restart session rồi Run All.')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError('Kaggle có nvidia-smi nhưng PyTorch không thấy CUDA. Hãy restart session sau khi bật GPU.')
    subprocess.run([nvidia_smi, '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
    print(f'CUDA OK: {torch.cuda.device_count()} GPU; pipeline dùng cuda:0')
require_kaggle_cuda()
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2-v5007',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v8-quality-repair-windowed-rerank',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'adapter_alias_policy': 'sha256-dedupe-v1',
    'source_alias_policy': 'sha256-dedupe-v1',
    'qlora_recovery_schema': 'legalqa-qlora-recovery-v1',
    'inference_pause_schema': 'inference-runtime-pause-v1',
    'offline_wheel_profile': 'hf-stack-2026-08-v1',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-5.14.1-', 'tokenizers-0.22.2-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'safetensors-0.8.0-', 'accelerate-1.14.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentence_transformers-5.7.0-', 'sentencepiece-0.2.1-', 'regex-2026.7.19-', 'click-8.4.2-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_selector_calibration.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
def source_identity(path):
    digest = hashlib.sha256()
    roots = ('SOURCE_CONTRACT.json', 'pyproject.toml', 'RELEASE_V9_2.md', 'configs', 'kaggle', 'scripts', 'src', 'tests', 'wheelhouse')
    files = []
    for name in roots:
        item = path / name
        files.extend([item] if item.is_file() else ([p for p in item.rglob('*') if p.is_file() and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts and p.suffix != '.pyc'] if item.is_dir() else []))
    for file in sorted(files, key=lambda p: p.relative_to(path).as_posix()):
        digest.update(file.relative_to(path).as_posix().encode('utf-8')); digest.update(b'\0')
        with file.open('rb') as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
source_groups = {}
for path in qualified_sources:
    source_groups.setdefault(source_identity(path), []).append(path)
if len(source_groups) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong QLoRA sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
source_aliases = next(iter(source_groups.values()))
selected_source = min(source_aliases, key=lambda p: (len(p.parts), p.as_posix()))
if len(source_aliases) > 1: print('Source aliases share one SHA-256 identity:', source_aliases)
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Môi trường source/offline wheels đã sẵn sàng.')


In [ ]:
recovery_candidates = sorted(input_root.rglob('selector_recovery_latest.zip')) if AUTO_RESUME else []
recovery_groups = {}
for path in recovery_candidates: recovery_groups.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), []).append(path)
if len(recovery_groups) > 1:
    raise RuntimeError(f'Có nhiều selector recovery ZIP: {recovery_candidates}')
command = [
    sys.executable, str(project / 'kaggle' / 'run_selector_calibration.py'),
    '--input-root', '/kaggle/input',
    '--working', WORKING,
    '--trials', str(TRIALS),
]
recovery_aliases = next(iter(recovery_groups.values())) if recovery_groups else []
if recovery_aliases:
    command.extend(['--resume-bundle', str(min(recovery_aliases, key=lambda p: (len(p.parts), p.as_posix())))])
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path(WORKING)
summary = json.loads((root / 'calibration_summary.json').read_text(encoding='utf-8'))
assert 300 <= summary['grouped_dev_count'] <= 500
assert (root / 'tuned_selection.yaml').is_file()
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Lưu tuned_selection.yaml thành Kaggle Dataset và attach vào notebook public v9.')
